##### Support Request Generator

Generate generic support requests from delivered orders and assign repeat user IDs.

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")
SUPPORT_RATE = float(dbutils.widgets.get("SUPPORT_RATE"))
LLM_MODEL = dbutils.widgets.get("LLM_MODEL")

In [ ]:
from pyspark.sql import functions as F

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.support")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.support.checkpoints")

# Deterministic repeatable user IDs and controlled edge-case distribution.
# - ~10% create an extra follow-up request for same order/user
# - ~3% create two extra follow-ups (spam/repeat bursts)
# - ~5% spammy/escalation phrasing
# - ~2% abusive phrasing
source = (
    spark.readStream
    .table(f"{CATALOG}.lakeflow.all_events")
    .filter(F.col("event_type") == "delivered")
    .filter(F.rand() < SUPPORT_RATE)
    .withColumn("seed", F.abs(F.hash(F.col("order_id"))))
    .withColumn("hash_bucket", F.col("seed") % F.lit(25))
    .withColumn("user_id", F.concat(F.lit("user-"), F.format_string("%03d", F.col("hash_bucket"))))
    .withColumn(
        "request_type",
        F.expr(
            """
            CASE
              WHEN hash_bucket % 5 = 0 THEN 'delivery_delay'
              WHEN hash_bucket % 5 = 1 THEN 'missing_items'
              WHEN hash_bucket % 5 = 2 THEN 'food_quality'
              WHEN hash_bucket % 5 = 3 THEN 'billing'
              ELSE 'general_support'
            END
            """
        ),
    )
    .withColumn(
        "edge_case_type",
        F.expr(
            """
            CASE
              WHEN seed % 100 < 2 THEN 'abusive_language'
              WHEN seed % 100 < 7 THEN 'spammy_escalation'
              WHEN seed % 100 < 15 THEN 'repeat_follow_up'
              WHEN seed % 100 < 21 THEN 'non_compensable_confusion'
              ELSE 'standard'
            END
            """
        ),
    )
    .withColumn(
        "repeat_multiplier",
        F.expr(
            """
            CASE
              WHEN seed % 100 < 3 THEN 3
              WHEN seed % 100 < 13 THEN 2
              ELSE 1
            END
            """
        ),
    )
)

expanded = (
    source
    .withColumn("repeat_idx", F.explode(F.sequence(F.lit(1), F.col("repeat_multiplier"))))
    .withColumn("support_request_id", F.expr("uuid()"))
    .withColumn("request_attempt", F.col("repeat_idx"))
    .withColumn(
        "request_text",
        F.expr(
            """
            CASE
              WHEN edge_case_type = 'abusive_language' THEN
                'This is unacceptable and ridiculous. Your team keeps messing this up. Fix this now.'
              WHEN edge_case_type = 'spammy_escalation' AND request_attempt > 1 THEN
                'Third message: still unresolved. Refund me immediately and escalate this to a supervisor now.'
              WHEN edge_case_type = 'spammy_escalation' THEN
                'Second request today. Please resolve this immediately. I need clear compensation details now.'
              WHEN edge_case_type = 'repeat_follow_up' AND request_attempt > 1 THEN
                'Following up again because this is not the first time this happened. Please review previous cases and offer a fair resolution.'
              WHEN request_type = 'delivery_delay' AND request_attempt > 1 THEN
                'This is another delay issue for me. I need a concrete resolution with refund and credit details.'
              WHEN request_type = 'delivery_delay' THEN
                'My delivery arrived much later than expected. Can you help with a credit or refund?'
              WHEN request_type = 'missing_items' AND request_attempt > 1 THEN
                'This is the second time items were missing from my order. Please make this right with a concrete offer.'
              WHEN request_type = 'missing_items' THEN
                'My order was missing one or more items. Please review and advise next steps.'
              WHEN request_type = 'food_quality' THEN
                'I had a quality issue with my food. I would like support and possible compensation.'
              WHEN request_type = 'billing' THEN
                'I think there is a billing issue on my recent order. Can you verify the charge?'
              WHEN edge_case_type = 'non_compensable_confusion' THEN
                'I changed my mind after ordering and wanted to check if I can still get a courtesy credit.'
              ELSE
                'I need help with a recent order. Please review my request and let me know what you can do.'
            END
            """
        ),
    )
    .select(
        "support_request_id",
        "user_id",
        "order_id",
        F.current_timestamp().alias("ts"),
        "request_type",
        "request_text",
        "edge_case_type",
        "request_attempt",
        F.lit("generator-v2").alias("generated_by"),
    )
)

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.support.raw_support_requests (
      support_request_id STRING,
      user_id STRING,
      order_id STRING,
      ts TIMESTAMP,
      request_type STRING,
      request_text STRING,
      edge_case_type STRING,
      request_attempt INT,
      generated_by STRING
    )
    """
)

# Backfill-safe schema evolution for existing environments.
try:
    spark.sql(f"ALTER TABLE {CATALOG}.support.raw_support_requests ADD COLUMNS (edge_case_type STRING)")
except Exception:
    pass

try:
    spark.sql(f"ALTER TABLE {CATALOG}.support.raw_support_requests ADD COLUMNS (request_attempt INT)")
except Exception:
    pass

expanded.writeStream \
    .option("checkpointLocation", f"/Volumes/{CATALOG}/support/checkpoints/support_request_generator") \
    .trigger(availableNow=True) \
    .table(f"{CATALOG}.support.raw_support_requests")

# Fallback batch seeding ensures fresh requests even when stream checkpoints have no new offsets.
spark.sql(
    f"""
    INSERT INTO {CATALOG}.support.raw_support_requests (
      support_request_id,
      user_id,
      order_id,
      ts,
      request_type,
      request_text,
      edge_case_type,
      request_attempt,
      generated_by
    )
    SELECT
      uuid() AS support_request_id,
      concat('user-', lpad(cast(abs(hash(order_id)) % 25 as string), 3, '0')) AS user_id,
      order_id,
      current_timestamp() AS ts,
      CASE
        WHEN abs(hash(order_id)) % 5 = 0 THEN 'delivery_delay'
        WHEN abs(hash(order_id)) % 5 = 1 THEN 'missing_items'
        WHEN abs(hash(order_id)) % 5 = 2 THEN 'food_quality'
        WHEN abs(hash(order_id)) % 5 = 3 THEN 'billing'
        ELSE 'general_support'
      END AS request_type,
      CASE
        WHEN abs(hash(order_id)) % 100 < 2 THEN 'This is unacceptable and keeps happening. Escalate this now.'
        WHEN abs(hash(order_id)) % 100 < 7 THEN 'Second request today: still unresolved. I need a concrete refund and credit now.'
        WHEN abs(hash(order_id)) % 100 < 15 THEN 'Following up again because this has happened before. Please offer a concrete resolution.'
        WHEN abs(hash(order_id)) % 5 = 1 THEN 'My order was missing items again. Please review history and make this right.'
        WHEN abs(hash(order_id)) % 5 = 0 THEN 'My delivery was delayed. Please provide a concrete refund and credit decision.'
        ELSE 'I need help with a recent order issue and would like a clear resolution.'
      END AS request_text,
      CASE
        WHEN abs(hash(order_id)) % 100 < 2 THEN 'abusive_language'
        WHEN abs(hash(order_id)) % 100 < 7 THEN 'spammy_escalation'
        WHEN abs(hash(order_id)) % 100 < 15 THEN 'repeat_follow_up'
        WHEN abs(hash(order_id)) % 100 < 21 THEN 'non_compensable_confusion'
        ELSE 'standard'
      END AS edge_case_type,
      CASE
        WHEN abs(hash(order_id)) % 100 < 3 THEN 3
        WHEN abs(hash(order_id)) % 100 < 13 THEN 2
        ELSE 1
      END AS request_attempt,
      'generator-v2-fallback-batch' AS generated_by
    FROM (
      SELECT DISTINCT order_id
      FROM (
        SELECT order_id FROM {CATALOG}.lakeflow.all_events WHERE event_type = 'delivered'
        UNION ALL
        SELECT order_id FROM {CATALOG}.support.support_agent_reports
      ) candidate_orders
      WHERE order_id IS NOT NULL
      ORDER BY order_id DESC
      LIMIT 50
    ) latest_delivered
    """
)